[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch03_wsl2_docker.ipynb)

# 第3章　WindowsでのAI開発環境構築：WSL2とDocker

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 3.2　WSL2のインストール

```powershell
wsl --install
```

## インストールが「成功したか」を確かめる

```powershell
wsl -l -v
```

```text
  NAME      STATE           VERSION
* Ubuntu    Running         2
```

```powershell
wsl --set-version Ubuntu 2
wsl --set-default-version 2   # 以後の既定を2にしておく
```

```bash
cat /etc/os-release | head -2   # Ubuntu 22.04 などと表示される
whoami                          # 初回設定で決めた自分のユーザー名
```

## 3.3　WSL2でGPUを使えるようにする

```bash
nvidia-smi
```

## 3.4　WSL2の設定を最適化する

```ini
[wsl2]
# WSL2に割り当てる最大メモリ
memory=48GB
# 割り当てるCPUコア数
processors=8
# スワップ領域
swap=16GB
```

## 3.5　まずはColabで動かす：環境構築の前に試す方法

In [ ]:
import torch
print(torch.cuda.is_available())      # True と出れば、GPUが使える
print(torch.cuda.get_device_name(0))  # 割り当てられたGPUの名前

## 3.7　Dockerのインストールと基本操作

```bash
docker ps                     # 起動中のコンテナ一覧
docker ps -a                  # 停止中も含む全コンテナ一覧
docker images                 # 手元にあるイメージ（コンテナの雛形）一覧
docker start dev-alice        # コンテナを起動
docker exec -it dev-alice bash    # 起動中のコンテナに入ってbashを使う
docker stop dev-alice         # コンテナを停止
```

```bash
docker run -it --name dev-alice \
  --gpus all \
  --cpus="6" \
  --memory="64g" \
  --shm-size="8g" \
  -v /home/alice/work:/workspace \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime bash
```

## 症状別トラブルシュート集：環境構築でつまずいたら

```bash
docker run --rm --gpus all pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime nvidia-smi
```

```bash
docker system df       # まず何がディスク容量を使用しているか確認
docker system prune -a # 使っていないイメージ・コンテナ・キャッシュを一括削除
```

```powershell
wsl --shutdown
diskpart
# diskpart内で以下を順に実行（パスは自分の環境に合わせる）
# select vdisk file="C:\Users\あなた\AppData\Local\...\ext4.vhdx"
# compact vdisk
```

## 演習

```bash
nvidia-smi                                   # ①WSL2（コンテナの外）でGPUが見えるか
docker run --rm --gpus all \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime \
  python -c "import torch; print(torch.cuda.is_available())"   # ②コンテナ内でTrueが出るか
```